# Research: Can `method=extract` work on `.xlsx`?

Hypothesis: our `sovExtractV1` analyzer fails on xlsx not because
Content Understanding can't process xlsx, but because of the specific
**`baseAnalyzerId`** / **config** / **field schema** combo we picked.

Evidence: calling `prebuilt-layout` and `prebuilt-documentSearch` directly
on an xlsx returns useful structured output.

**Test matrix** (Acme xlsx as the fixed input):

| # | baseAnalyzerId | method | config flags | Notes |
|---|---|---|---|---|
| 1 | `prebuilt-layout` (no fields) | n/a | default | Confirms layout works |
| 2 | `prebuilt-documentSearch` (no fields) | n/a | default | Confirms doc search works |
| 3 | `prebuilt-document` + our SOV fields, `method=extract` | extract | enableOcr/enableLayout/estimateFieldSourceAndConfidence | Current `sovExtractV1` -- baseline (expect empty) |
| 4 | `prebuilt-layout` + our SOV fields, `method=extract` | extract | default | Switch base only |
| 5 | `prebuilt-documentSearch` + our SOV fields, `method=extract` | extract | default | Switch base only |
| 6 | `prebuilt-layout` + our SOV fields, `method=generate` | generate | default | Maybe layout+generate is the intended combo for xlsx |
| 7 | `prebuilt-document` + minimal flags | extract | only `returnDetails` | Strip config, see if any flag is hostile |

We'll record per call: `_meta.elapsed_sec`, count of fields with
non-null values, and a peek at `markdown` / `contents[0]` keys so we
can see whether layout/markdown surfaced anything.

In [6]:
from __future__ import annotations
import json, os, time, uuid
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential

# Walk up from cwd to find the repo root (folder containing demo/sov or .git)
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'demo' / 'sov').exists() or (p / '.git').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from {start}')

REPO = _find_repo_root(Path.cwd().resolve())
print('REPO:', REPO)

# Prefer repo-root .env; fall back to apps/workshop/api/.env
for _cand in [REPO / '.env', REPO / 'apps' / 'workshop' / 'api' / '.env']:
    if _cand.exists():
        load_dotenv(_cand)
        print('env:', _cand)
        break

endpoint = os.environ.get('APP_CONTENT_UNDERSTANDING_ENDPOINT') or os.environ.get('CONTENTUNDERSTANDING_ENDPOINT')
print('endpoint:', endpoint)
if not endpoint:
    raise RuntimeError('Missing APP_CONTENT_UNDERSTANDING_ENDPOINT in .env')

DEMO = REPO / 'demo' / 'sov'
ATTACH = DEMO / 'attachments'
TPL_DIR = DEMO / 'reference' / 'analyzer-templates'
OUT = REPO / 'feedback' / 'underwriting' / 'research-output'
OUT.mkdir(parents=True, exist_ok=True)

# Sanity check that paths actually exist before we proceed
assert TPL_DIR.is_dir(), f'TPL_DIR missing: {TPL_DIR}'
assert ATTACH.is_dir(), f'ATTACH missing: {ATTACH}'

TARGET = ATTACH / '01_acme_SOV.xlsx'
CONTENT_TYPE = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
assert TARGET.exists(), f'TARGET missing: {TARGET}'

client = ContentUnderstandingClient(endpoint=endpoint, credential=DefaultAzureCredential())
print('client OK')


REPO: C:\Users\jomedin\Documents\azure-ai-foundry-insurance
env: C:\Users\jomedin\Documents\azure-ai-foundry-insurance\.env
endpoint: https://ais-aidemos-dev-01.services.ai.azure.com/
client OK


In [7]:
# Load our existing SOV field schema so we can plug it into different bases.
with open(TPL_DIR / 'sov_extraction.json', 'r', encoding='utf-8') as f:
    sov_extract_tpl = json.load(f)
with open(TPL_DIR / 'sov_extraction_generate.json', 'r', encoding='utf-8') as f:
    sov_generate_tpl = json.load(f)

# Force every field's method to a specific value (for variant 4-7 tests).
def with_method(template: dict, method: str) -> dict:
    t = json.loads(json.dumps(template))
    def walk(node):
        if isinstance(node, dict):
            if 'method' in node and isinstance(node.get('method'), str):
                node['method'] = method
            for v in node.values(): walk(v)
        elif isinstance(node, list):
            for v in node: walk(v)
    walk(t)
    return t

sov_fields = sov_extract_tpl['fieldSchema']
print('schema fields:', list(sov_fields['fields'].keys())[:8], '...')

schema fields: ['InsuredName', 'DBA', 'MailingAddress', 'EffectiveDate', 'ExpirationDate', 'PrimaryOperations', 'NAICS', 'Currency'] ...


In [12]:
def _result_to_dict(r):
    if hasattr(r, 'as_dict'): return r.as_dict()
    return json.loads(json.dumps(r, default=lambda o: getattr(o, '__dict__', str(o))))

def _safe_aid(name: str) -> str:
    # CU rejects '-' in analyzerId; use underscores + alnum only.
    cleaned = ''.join(ch if (ch.isalnum() or ch == '_') else '_' for ch in name)
    return f"research_{cleaned}_{uuid.uuid4().hex[:6]}"

def ensure_analyzer(aid: str, body: dict):
    try:
        client.delete_analyzer(analyzer_id=aid)
    except Exception:
        pass
    poller = client.begin_create_analyzer(analyzer_id=aid, resource=body)
    poller.result()

def _summarize(name: str, base: str, has_fields: bool, payload: dict, elapsed: float, raw_path: Path, error: str | None = None) -> dict:
    contents = payload.get('contents', []) if isinstance(payload, dict) else []
    c0 = contents[0] if contents else {}
    fields = c0.get('fields', {}) if isinstance(c0, dict) else {}
    populated = sum(1 for k, v in fields.items() if isinstance(v, dict) and any(
        kk.startswith('value') and v.get(kk) not in (None, '', [], {}) for kk in v
    ))
    md = c0.get('markdown') or '' if isinstance(c0, dict) else ''
    return {
        'variant': name,
        'base': base,
        'has_fields': has_fields,
        'elapsed_sec': elapsed,
        'contents_keys': list(c0.keys())[:8] if isinstance(c0, dict) else [],
        'field_count': len(fields),
        'populated_top_level_fields': populated,
        'markdown_chars': len(md),
        'markdown_preview': md[:200],
        'raw_path': str(raw_path.relative_to(REPO)) if raw_path.exists() else '',
        'error': error,
    }

def run_existing(name: str, analyzer_id: str, target: Path = TARGET, content_type: str = CONTENT_TYPE) -> dict:
    """Call an existing (built-in) analyzer directly -- no create/delete."""
    print(f"\n=== {name}  analyzer={analyzer_id}  (existing, no create) ===")
    raw_path = OUT / f"{name}.json"
    t0 = time.perf_counter()
    try:
        with open(target, 'rb') as f:
            poller = client.begin_analyze_binary(analyzer_id=analyzer_id, binary_input=f.read(), content_type=content_type)
        payload = _result_to_dict(poller.result())
        elapsed = round(time.perf_counter() - t0, 2)
        raw_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        return _summarize(name, analyzer_id, has_fields=False, payload=payload, elapsed=elapsed, raw_path=raw_path)
    except Exception as e:
        elapsed = round(time.perf_counter() - t0, 2)
        print(f"  ! analyze failed: {e}")
        return _summarize(name, analyzer_id, has_fields=False, payload={}, elapsed=elapsed, raw_path=raw_path, error=str(e)[:300])

def run_variant(name: str, body: dict, target: Path = TARGET, content_type: str = CONTENT_TYPE) -> dict:
    """Create an ad-hoc custom analyzer, call analyze, return summary stats + raw payload path.
    Catches creation failures (e.g. InvalidBaseAnalyzerId) so the rest of the matrix keeps running."""
    aid = _safe_aid(name)
    base = body.get('baseAnalyzerId')
    print(f"\n=== {name}  base={base}  fields={'fieldSchema' in body} ===")
    raw_path = OUT / f"{name}.json"
    t0 = time.perf_counter()
    try:
        ensure_analyzer(aid, body)
    except Exception as e:
        elapsed = round(time.perf_counter() - t0, 2)
        print(f"  ! create_analyzer failed: {e}")
        return _summarize(name, base, has_fields='fieldSchema' in body, payload={}, elapsed=elapsed, raw_path=raw_path, error=str(e)[:300])
    try:
        with open(target, 'rb') as f:
            poller = client.begin_analyze_binary(analyzer_id=aid, binary_input=f.read(), content_type=content_type)
        payload = _result_to_dict(poller.result())
        elapsed = round(time.perf_counter() - t0, 2)
        raw_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        return _summarize(name, base, has_fields='fieldSchema' in body, payload=payload, elapsed=elapsed, raw_path=raw_path)
    except Exception as e:
        elapsed = round(time.perf_counter() - t0, 2)
        print(f"  ! analyze failed: {e}")
        return _summarize(name, base, has_fields='fieldSchema' in body, payload={}, elapsed=elapsed, raw_path=raw_path, error=str(e)[:300])
    finally:
        try:
            client.delete_analyzer(analyzer_id=aid)
        except Exception:
            pass

results = []


In [13]:
# --- Variant 1: prebuilt-layout, called directly (sanity check, no creation)
results.append(run_existing('01_layout_only', 'prebuilt-layout'))
results[-1]



=== 01_layout_only  analyzer=prebuilt-layout  (existing, no create) ===


{'variant': '01_layout_only',
 'base': 'prebuilt-layout',
 'has_fields': False,
 'elapsed_sec': 3.42,
 'contents_keys': ['path',
  'markdown',
  'kind',
  'startPageNumber',
  'endPageNumber',
  'analyzerId',
  'mimeType'],
 'field_count': 0,
 'populated_top_level_fields': 0,
 'markdown_chars': 12380,
 'markdown_preview': '# SOV\n\n| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |',
 'raw_path': 'feedback\\underwriting\\research-output\\01_layout_only.json',
 'error': None}

In [14]:
# --- Variant 2: prebuilt-documentSearch, called directly (no creation)
results.append(run_existing('02_docsearch_only', 'prebuilt-documentSearch'))
results[-1]



=== 02_docsearch_only  analyzer=prebuilt-documentSearch  (existing, no create) ===


{'variant': '02_docsearch_only',
 'base': 'prebuilt-documentSearch',
 'has_fields': False,
 'elapsed_sec': 6.69,
 'contents_keys': ['path',
  'markdown',
  'fields',
  'kind',
  'startPageNumber',
  'endPageNumber',
  'analyzerId',
  'mimeType'],
 'field_count': 1,
 'populated_top_level_fields': 1,
 'markdown_chars': 12380,
 'markdown_preview': '# SOV\n\n| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |',
 'raw_path': 'feedback\\underwriting\\research-output\\02_docsearch_only.json',
 'error': None}

In [15]:
# --- Variant 3: current sovExtractV1 (baseline)  (prebuilt-document + extract + flags)
results.append(run_variant('03_baseline_sovExtract', sov_extract_tpl))
results[-1]


=== 03_baseline_sovExtract  base=prebuilt-document  fields=True ===


{'variant': '03_baseline_sovExtract',
 'base': 'prebuilt-document',
 'has_fields': True,
 'elapsed_sec': 23.32,
 'contents_keys': ['path',
  'markdown',
  'fields',
  'kind',
  'startPageNumber',
  'endPageNumber',
  'analyzerId',
  'mimeType'],
 'field_count': 18,
 'populated_top_level_fields': 2,
 'markdown_chars': 12380,
 'markdown_preview': '# SOV\n\n| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |',
 'raw_path': 'feedback\\underwriting\\research-output\\03_baseline_sovExtract.json',
 'error': None}

In [16]:
# --- Variant 4: switch base to prebuilt-layout, keep extract method
v4 = json.loads(json.dumps(sov_extract_tpl))
v4['baseAnalyzerId'] = 'prebuilt-layout'
results.append(run_variant('04_layout_extract', v4))
results[-1]


=== 04_layout_extract  base=prebuilt-layout  fields=True ===
  ! create_analyzer failed: (InvalidRequest) Invalid Request.
Code: InvalidRequest
Message: Invalid Request.
Inner error: {
    "code": "InvalidFieldSchema",
    "message": "One or more errors encountered while processing the field schema.",
    "details": [
        {
            "code": "InvalidBaseAnalyzerId",
            "message": "Unsupported 'baseAnalyzerId' value: 'prebuilt-layout'",
            "target": "/baseAnalyzerId"
        }
    ]
}


{'variant': '04_layout_extract',
 'base': 'prebuilt-layout',
 'has_fields': True,
 'elapsed_sec': 0.21,
 'contents_keys': [],
 'field_count': 0,
 'populated_top_level_fields': 0,
 'markdown_chars': 0,
 'markdown_preview': '',
 'raw_path': '',
 'error': '(InvalidRequest) Invalid Request.\nCode: InvalidRequest\nMessage: Invalid Request.\nInner error: {\n    "code": "InvalidFieldSchema",\n    "message": "One or more errors encountered while processing the field schema.",\n    "details": [\n        {\n            "code": "InvalidBaseAnalyzerId",\n            "m'}

In [17]:
# --- Variant 5: switch base to prebuilt-documentSearch, keep extract method
v5 = json.loads(json.dumps(sov_extract_tpl))
v5['baseAnalyzerId'] = 'prebuilt-documentSearch'
results.append(run_variant('05_docsearch_extract', v5))
results[-1]


=== 05_docsearch_extract  base=prebuilt-documentSearch  fields=True ===
  ! create_analyzer failed: (InvalidRequest) Invalid Request.
Code: InvalidRequest
Message: Invalid Request.
Inner error: {
    "code": "InvalidFieldSchema",
    "message": "One or more errors encountered while processing the field schema.",
    "details": [
        {
            "code": "InvalidBaseAnalyzerId",
            "message": "Unsupported 'baseAnalyzerId' value: 'prebuilt-documentSearch'",
            "target": "/baseAnalyzerId"
        }
    ]
}


{'variant': '05_docsearch_extract',
 'base': 'prebuilt-documentSearch',
 'has_fields': True,
 'elapsed_sec': 0.24,
 'contents_keys': [],
 'field_count': 0,
 'populated_top_level_fields': 0,
 'markdown_chars': 0,
 'markdown_preview': '',
 'raw_path': '',
 'error': '(InvalidRequest) Invalid Request.\nCode: InvalidRequest\nMessage: Invalid Request.\nInner error: {\n    "code": "InvalidFieldSchema",\n    "message": "One or more errors encountered while processing the field schema.",\n    "details": [\n        {\n            "code": "InvalidBaseAnalyzerId",\n            "m'}

In [18]:
# --- Variant 6: prebuilt-layout + force every field method=generate
v6 = with_method(sov_extract_tpl, 'generate')
v6['baseAnalyzerId'] = 'prebuilt-layout'
results.append(run_variant('06_layout_generate', v6))
results[-1]


=== 06_layout_generate  base=prebuilt-layout  fields=True ===
  ! create_analyzer failed: (InvalidRequest) Invalid Request.
Code: InvalidRequest
Message: Invalid Request.
Inner error: {
    "code": "InvalidFieldSchema",
    "message": "One or more errors encountered while processing the field schema.",
    "details": [
        {
            "code": "InvalidBaseAnalyzerId",
            "message": "Unsupported 'baseAnalyzerId' value: 'prebuilt-layout'",
            "target": "/baseAnalyzerId"
        }
    ]
}


{'variant': '06_layout_generate',
 'base': 'prebuilt-layout',
 'has_fields': True,
 'elapsed_sec': 0.21,
 'contents_keys': [],
 'field_count': 0,
 'populated_top_level_fields': 0,
 'markdown_chars': 0,
 'markdown_preview': '',
 'raw_path': '',
 'error': '(InvalidRequest) Invalid Request.\nCode: InvalidRequest\nMessage: Invalid Request.\nInner error: {\n    "code": "InvalidFieldSchema",\n    "message": "One or more errors encountered while processing the field schema.",\n    "details": [\n        {\n            "code": "InvalidBaseAnalyzerId",\n            "m'}

In [19]:
# --- Variant 7: prebuilt-document, minimal config (drop OCR/Layout/grounding flags)
v7 = json.loads(json.dumps(sov_extract_tpl))
v7['config'] = {'returnDetails': True}
results.append(run_variant('07_min_config', v7))
results[-1]


=== 07_min_config  base=prebuilt-document  fields=True ===


{'variant': '07_min_config',
 'base': 'prebuilt-document',
 'has_fields': True,
 'elapsed_sec': 26.78,
 'contents_keys': ['path',
  'markdown',
  'fields',
  'kind',
  'startPageNumber',
  'endPageNumber',
  'analyzerId',
  'mimeType'],
 'field_count': 18,
 'populated_top_level_fields': 2,
 'markdown_chars': 12380,
 'markdown_preview': '# SOV\n\n| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |',
 'raw_path': 'feedback\\underwriting\\research-output\\07_min_config.json',
 'error': None}

## Comparison

Higher `populated_top_level_fields` = the schema actually got values back.
Look at `markdown_chars` to see whether layout extracted anything from the
workbook even when the field schema came back empty.

In [20]:
df = pd.DataFrame(results)
df_view = df[['variant', 'base', 'has_fields', 'field_count', 'populated_top_level_fields', 'markdown_chars', 'elapsed_sec', 'error']]
df_view


,variant,base,has_fields,field_count,populated_top_level_fields,markdown_chars,elapsed_sec,error
0,01_layout_only,prebuilt-layout,False,0,0,12380,3.42,NaN
1,02_docsearch_only,prebuilt-documentSearch,False,1,1,12380,6.69,NaN
2,03_baseline_sovExtract,prebuilt-document,True,18,2,12380,23.32,NaN
3,04_layout_extract,prebuilt-layout,True,0,0,0,0.21,(InvalidRequest) Invalid Request.\nCode: Inval...
4,05_docsearch_extract,prebuilt-documentSearch,True,0,0,0,0.24,(InvalidRequest) Invalid Request.\nCode: Inval...
5,06_layout_generate,prebuilt-layout,True,0,0,0,0.21,(InvalidRequest) Invalid Request.\nCode: Inval...
6,07_min_config,prebuilt-document,True,18,2,12380,26.78,NaN


In [21]:
# Markdown previews -- helps you see what each base actually pulled out of the xlsx.
for r in results:
    print(f"\n=== {r['variant']}  base={r['base']}  populated={r['populated_top_level_fields']}")
    print(r['markdown_preview'] or '(no markdown)')


=== 01_layout_only  base=prebuilt-layout  populated=0
# SOV

| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |

=== 02_docsearch_only  base=prebuilt-documentSearch  populated=1
# SOV

| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |

=== 03_baseline_sovExtract  base=prebuilt-document  populated=2
# SOV

| STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. | STATEMENT OF VALUES — Acme Manufacturing & Distribution, Inc. |

=== 04_layout_extract  base=prebuilt-layout  populated=0
(no markdown)

=== 05_docsearch_extract  base=prebuilt-documentSearch  populated=0
(no markdown)

=== 06_layout_generate  base=prebuilt-layout  populated

In [22]:
# Drill into which fields actually got values for the variants that produced output.
def populated_fields(raw_path: str) -> dict:
    p = REPO / raw_path
    if not p.exists(): return {}
    payload = json.loads(p.read_text(encoding='utf-8'))
    fields = (payload.get('contents') or [{}])[0].get('fields', {})
    out = {}
    for k, v in fields.items():
        if not isinstance(v, dict): continue
        # Look for any valueX with a real value
        vals = {kk: vv for kk, vv in v.items() if kk.startswith('value') and vv not in (None, '', [], {})}
        if vals:
            preview = next(iter(vals.values()))
            if isinstance(preview, list):
                preview = f"[{len(preview)} items] first={preview[0] if preview else None}"
            out[k] = {'type': v.get('type'), 'method': v.get('method'), 'confidence': v.get('confidence'), 'value_preview': str(preview)[:120]}
    return out

for r in results:
    if not r.get('raw_path'): continue
    pop = populated_fields(r['raw_path'])
    print(f"\n=== {r['variant']}  populated_fields={len(pop)}")
    for fname, info in pop.items():
        print(f"  - {fname}: type={info['type']} method={info['method']} conf={info['confidence']} value={info['value_preview']}")



=== 01_layout_only  populated_fields=0

=== 02_docsearch_only  populated_fields=1
  - Summary: type=string method=None conf=1 value=This document is a Statement of Values for Acme Manufacturing & Distribution, Inc., detailing insured values, locations,

=== 03_baseline_sovExtract  populated_fields=2
  - Currency: type=string method=None conf=1 value=USD
  - Locations: type=array method=None conf=None value=[19 items] first={'type': 'object', 'valueObject': {'LocationNumber': {'type': 'number', 'confidence': 1}, 'BuildingNumb

=== 07_min_config  populated_fields=2
  - Currency: type=string method=None conf=1 value=USD
  - Locations: type=array method=None conf=None value=[19 items] first={'type': 'object', 'valueObject': {'LocationNumber': {'type': 'number', 'confidence': 1}, 'BuildingNumb


In [24]:
# Inspect the Locations array from variant 3 -- is the data real or just stubs?
v3 = json.loads((REPO / 'feedback' / 'underwriting' / 'research-output' / '03_baseline_sovExtract.json').read_text(encoding='utf-8'))
locs = v3['contents'][0]['fields']['Locations']
items = locs.get('valueArray', [])
print(f"Locations: {len(items)} items")

# Show first location only, compactly
if items:
    obj = items[0].get('valueObject', {})
    print("\nFirst Location -- populated subfields:")
    for fname, fval in obj.items():
        v = next((vv for kk, vv in fval.items() if kk.startswith('value')), None)
        if v not in (None, '', [], {}):
            c = fval.get('confidence')
            print(f"  {fname:24s} = {str(v)[:50]:50s}  conf={c}")
    empty_subfields = [k for k, fv in obj.items()
                       if not any(kk.startswith('value') and fv.get(kk) not in (None, '', [], {}) for kk in fv)]
    print(f"\n  Empty subfields ({len(empty_subfields)}): {empty_subfields[:8]}{' ...' if len(empty_subfields) > 8 else ''}")

# Top-level scalars that came back empty
empty_scalars = []
for k, v in v3['contents'][0]['fields'].items():
    if isinstance(v, dict) and v.get('type') in ('string', 'number', 'date', 'integer'):
        has_val = any(kk.startswith('value') and v.get(kk) not in (None, '', [], {}) for kk in v)
        if not has_val:
            empty_scalars.append(k)
print(f"\nEmpty top-level scalars ({len(empty_scalars)}): {empty_scalars}")


Locations: 19 items

First Location -- populated subfields:

  Empty subfields (22): ['LocationNumber', 'BuildingNumber', 'Street', 'City', 'State', 'Zip', 'ConstructionType', 'Occupancy'] ...

Empty top-level scalars (16): ['InsuredName', 'DBA', 'MailingAddress', 'EffectiveDate', 'ExpirationDate', 'PrimaryOperations', 'NAICS', 'ValuationDate', 'TotalInsuredValue', 'LocationCount', 'BrokerName', 'BrokerContact', 'BrokerEmail', 'BrokerPhone', 'PreparedBy', 'PreparedDate']


In [27]:
# Pretty-print variant 3 (baseline sovExtractV1) — full account header + all 19 locations.
v3 = json.loads((REPO / 'feedback' / 'underwriting' / 'research-output' / '03_baseline_sovExtract.json').read_text(encoding='utf-8'))
fields = v3['contents'][0]['fields']

def _val(node):
    if not isinstance(node, dict): return None, None
    for kk, vv in node.items():
        if kk.startswith('value') and vv not in (None, '', [], {}):
            return vv, node.get('confidence')
    return None, node.get('confidence')

# --- Account header ---
print('=' * 78)
print(' ACCOUNT HEADER (top-level scalars)')
print('=' * 78)
header_keys = [k for k, v in fields.items() if isinstance(v, dict) and v.get('type') in ('string', 'number', 'date', 'integer')]
for k in header_keys:
    val, conf = _val(fields[k])
    val_str = '<empty>' if val in (None, '', [], {}) else str(val)
    conf_str = f'{conf:.2f}' if isinstance(conf, (int, float)) else '   -'
    marker = ' ' if val_str != '<empty>' else 'x'
    print(f'  [{marker}] {k:22s}  conf={conf_str}  {val_str[:60]}')

# --- Locations ---
loc_items = (fields.get('Locations') or {}).get('valueArray', [])
print()
print('=' * 78)
print(f' LOCATIONS  ({len(loc_items)} rows detected)')
print('=' * 78)

if loc_items:
    # Get the union of subfield names from the first row
    subfield_names = list((loc_items[0].get('valueObject') or {}).keys())
    # Header
    print(f'  {"#":>2}  ' + '  '.join(f'{n[:12]:12s}' for n in subfield_names[:6]) + '  ...')
    print('  ' + '-' * 76)
    for i, item in enumerate(loc_items, 1):
        obj = item.get('valueObject') or {}
        cells = []
        for n in subfield_names[:6]:
            val, _ = _val(obj.get(n, {}))
            cells.append(f'{(str(val) if val not in (None, "", [], {}) else "-")[:12]:12s}')
        print(f'  {i:>2}  ' + '  '.join(cells))

    # How many of (subfields x rows) actually populated?
    total = len(loc_items) * len(subfield_names)
    populated = 0
    for item in loc_items:
        obj = item.get('valueObject') or {}
        for n in subfield_names:
            val, _ = _val(obj.get(n, {}))
            if val not in (None, '', [], {}):
                populated += 1
    print()
    print(f'  Populated location cells: {populated} / {total}  ({populated * 100 // total}%)')


 ACCOUNT HEADER (top-level scalars)
  [x] InsuredName             conf=1.00  <empty>
  [x] DBA                     conf=1.00  <empty>
  [x] MailingAddress          conf=1.00  <empty>
  [x] EffectiveDate           conf=1.00  <empty>
  [x] ExpirationDate          conf=1.00  <empty>
  [x] PrimaryOperations       conf=1.00  <empty>
  [x] NAICS                   conf=1.00  <empty>
  [ ] Currency                conf=1.00  USD
  [x] ValuationDate           conf=1.00  <empty>
  [x] TotalInsuredValue       conf=1.00  <empty>
  [x] LocationCount           conf=1.00  <empty>
  [x] BrokerName              conf=1.00  <empty>
  [x] BrokerContact           conf=1.00  <empty>
  [x] BrokerEmail             conf=1.00  <empty>
  [x] BrokerPhone             conf=1.00  <empty>
  [x] PreparedBy              conf=1.00  <empty>
  [x] PreparedDate            conf=1.00  <empty>

 LOCATIONS  (19 rows detected)
   #  LocationNumb  BuildingNumb  Street        City          State         Zip           ...
  -------

## Findings

### Empirical results (Acme xlsx)
- **`prebuilt-layout`** (called directly): full markdown extraction, 12,380 chars.
- **`prebuilt-documentSearch`** (called directly): same markdown + a `Summary` field.
- **`prebuilt-document` + custom field schema, `method=extract`** (variants 3, 7):
  - Detects array cardinality: `Locations` array correctly returned with **19 items**.
  - Resolves a single deterministic top-level scalar: `Currency: USD`.
  - **Every other value is empty.** All 16 header scalars and all 22×19 = 418 location subfields come back with `confidence: 1` but no `valueX`.
  - True regardless of full config vs. minimal `returnDetails`-only config.
- **`prebuilt-layout` / `prebuilt-documentSearch` as `baseAnalyzerId`** (variants 4, 5, 6): API rejects with `InvalidBaseAnalyzerId`.

### This is documented behavior — not a bug

**1. Office files (.xlsx, .docx, .pptx) only get "Minimal" processing.**
From [Service quotas and limits → Input file limits](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/service-limits#input-file-limits):

| Format | Processing |
|---|---|
| `.pdf, .tiff, .jpg, .png, .bmp, .heif, .heic` | **Basic (OCR) or Standard (Layout)** |
| `.docx, .xlsx, .pptx` | **Minimal** |
| `.txt, .html, .md, .rtf, .eml, .msg, .xml` | Minimal |

Only PDFs and images go through the OCR / layout pipeline. xlsx text is read directly, with no layout grounding.

**2. `method=extract` is a grounded extraction method.**
From [Document overview → Field extraction methods](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/document/overview#field-extraction-methods):
- *Extract*: "Extract specific data ... for precise and focused information capture." — relies on `estimateFieldSourceAndConfidence`, which needs OCR/layout spans.
- *Generate*: "Produce new insights or summaries ..." — LLM over the markdown, no layout dependency.

Since xlsx skips the layout pipeline, `extract` has nothing to ground against. The service returns the schema shell (confidence on the *decision that the field is absent*, not on a value) but no `valueX`. The single populated value, `Currency: USD`, is a literal token that survives without grounding.

**3. Only four analyzers can be used as `baseAnalyzerId`.**
From [Prebuilt analyzers → Base analyzers](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/prebuilt-analyzers#base-analyzers):

> Currently, you can only derive custom analyzers from this set of four base analyzers: `prebuilt-audio`, `prebuilt-document`, `prebuilt-image`, `prebuilt-video`.

That explains the `InvalidBaseAnalyzerId` errors for variants 4–6.

**4. The PDF-only-rich-features pattern is consistent across the docs.**
- `prebuilt-layout`: "Detects figure types ... (**PDF files only**)."
- `prebuilt-documentSearch`: "Figure analysis is only supported for **PDF and image file formats**."

Same idea: layout/figure-grounded features are PDF/image-only; Office formats fall through to the Minimal text path.

### Conclusion
For xlsx, `method=extract` is **structurally not viable** — not a configuration bug. There is no `baseAnalyzerId` swap that fixes it because the only legal bases for custom field schemas are `prebuilt-document` (no layout for xlsx) and the non-document modalities. The intended path for xlsx is `method=generate`, which feeds the layout-derived markdown to the LLM — exactly what `sovGenerateV1` (patterns B and C) does, and why those patterns work where Pattern A doesn't.

### Implication for the workshop app
- Pattern A (`extract`) should be marked as **PDF-recommended**. Workshop attendees who select Pattern A on an xlsx will see the same near-empty result we did, with `confidence: 1` everywhere — that's deceptive and worth a warning in the UI.
- Patterns B and C (`generate`) remain the correct choice for xlsx and continue to support PDFs as well.

Raw payloads for all 7 variants are in [feedback/underwriting/research-output/](research-output/).
